In [0]:
# ==============================================================================
# CAMADA GOLD: Modelagem Dimensional (Star Schema) & Metadados
# ==============================================================================

from pyspark.sql.functions import col, dense_rank, current_timestamp
from pyspark.sql.window import Window

# 1. Leitura da Tabela Delta Silver
df_silver = spark.table("workspace.default.silver_listings")

print("Iniciando a criação do Star Schema na Camada Gold...")

# ------------------------------------------------------------------------------
# DIMENSÃO ANFITRIÃO (dim_host)
# ------------------------------------------------------------------------------
df_dim_host = df_silver.select(
    "host_id",
    "host_name",
    "host_since",
    "is_superhost",
    "host_listings_count"
).distinct()

df_dim_host.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_host")

# ------------------------------------------------------------------------------
# DIMENSÃO LOCALIZAÇÃO (dim_location)
# ------------------------------------------------------------------------------
# Gerando ID único para cada combinação de Bairro e Zona
window_loc = Window.orderBy("neighbourhood", "zone")
df_dim_location = df_silver.select("neighbourhood", "zone").distinct() \
    .withColumn("location_id", dense_rank().over(window_loc)) \
    .select("location_id", "neighbourhood", "zone")

df_dim_location.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_location")

# ------------------------------------------------------------------------------
# DIMENSÃO IMÓVEL E COMODIDADES (dim_property)
# ------------------------------------------------------------------------------
df_dim_property = df_silver.select(
    col("listing_id").alias("property_id"),
    "property_type",
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "has_wifi",
    "has_air_conditioning",
    "has_pool",
    "has_sea_view"
).distinct()

df_dim_property.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_property")

# ------------------------------------------------------------------------------
# TABELA FATO (fact_listings)
# ------------------------------------------------------------------------------
# Cruzamento com dim_location para recuperar a Foreign Key location_id
df_fact = df_silver.join(df_dim_location, on=["neighbourhood", "zone"], how="inner") \
    .select(
        col("listing_id").alias("property_id"),
        "host_id",
        "location_id",
        "price",
        "minimum_nights",
        "maximum_nights",
        "number_of_reviews",
        "review_score",
        col("latitude"),
        col("longitude"),
        current_timestamp().alias("_created_at")
    )

df_fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.fact_listings")

print("✅ Tabelas da Camada Gold criadas com sucesso!")

In [0]:
%sql
-- Documentação do Catálogo de Dados para a Tabela Fato
COMMENT ON TABLE workspace.default.fact_listings IS 'Tabela Fato contendo as métricas de preço e regras dos anúncios de aluguel no RJ.';

COMMENT ON COLUMN workspace.default.fact_listings.property_id IS 'Chave Estrangeira (FK) apontando para dim_property.';
COMMENT ON COLUMN workspace.default.fact_listings.host_id IS 'Chave Estrangeira (FK) apontando para dim_host.';
COMMENT ON COLUMN workspace.default.fact_listings.location_id IS 'Chave Estrangeira (FK) apontando para dim_location.';
COMMENT ON COLUMN workspace.default.fact_listings.price IS 'Valor em BRL da diária do anúncio (Decimal 10,2).';